In [1]:
import pandas as pd
import os
import sys
sys.path.append('../src')

from preprocessing import get_dfs, create_static_df, create_medication_df

In [2]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)

In [3]:
medication = create_medication_df(dfs)
medication

Unique patients in medication: 3335
Removing patients that are not in static_df
Unique patients in clinical assessments: 3296
Average entries per patient 78.9


,patient_id,transplant_id,description,ddd,unit,atc,start,end
0,8905,9891,AEB071,200,mg,XXXXXXX,1,4
1,8905,9891,AEB071,200,mg,XXXXXXX,155,155
2,8905,9891,AEB071,400,mg,XXXXXXX,6,154
3,8905,9891,AEB071,300,mg,XXXXXXX,5,5
4,9086,10085,AEB071,400,mg,XXXXXXX,5,361
...,...,...,...,...,...,...,...,...
260213,14569,13283,Thymoglobuline,75,mg,L04AA04,0,1
260214,14656,13133,Thymoglobuline,150,mg,L04AA04,0,1
260215,12673,11660,Thymoglobuline,100,mg,L04AA04,0,1
260216,13684,12383,Thymoglobuline,125,mg,L04AA04,0,1


In [4]:
print(dfs['medikation'].columns)
dfs['medikation'].count()

Index(['MedikationID', 'PatientID', 'TransplantationID', 'prescription start',
       'prescription end', 'Bezeichnung', 'DDD', 'unit', 'ATC'],
      dtype='object')


MedikationID          266897
PatientID             266897
TransplantationID     266897
prescription start    266897
prescription end      252160
Bezeichnung           266897
DDD                   266016
unit                  263723
ATC                   266897
dtype: int64

In [8]:
print(len(dfs['medikation']))
dfs['medikation']['Bezeichnung'].value_counts().head(10)

266897


Bezeichnung
Tacrolimus              112517
Methylprednisolon        74425
Ciclosporin              29579
Mycophenolatmofetil      11562
Mycophenolat Mofetil      5487
Everolimus                5463
Mycophenolatnatrium       5443
Mycophenolsäure           3404
Prednisolon               3176
Basiliximab               2002
Name: count, dtype: int64

In [5]:
medication = dfs['medikation'][['PatientID', 'TransplantationID', 'prescription start', 'prescription end', 'Bezeichnung', 'DDD', 'unit', 'ATC']].rename(
        columns={
            'PatientID': 'patient_id',
            'TransplantationID': 'transplant_id',
            'prescription start': 'p_start',
            'prescription end': 'p_end',
            'Bezeichnung': 'description',
            'DDD': 'ddd', # defined daily dose
            'unit': 'unit',
            'ATC': 'atc' # the ATC column identifies the medication based on its pharmacological classification.
        }
    )

print(f"Unique patients in medication: {medication['patient_id'].nunique()}")
print('Removing patients that are not in static_df')
medication = medication.merge(
    static_df[['patient_id', 'transplant_id', 'transplant_date']],
    how='inner',  # Inner join to keep only matching entries
    on=['patient_id', 'transplant_id']
)

# Calculate the relative days from transplantation date to prescription start and end
medication['start'] = (pd.to_datetime(medication['p_start'], dayfirst=True)-
                                             pd.to_datetime(medication['transplant_date'], dayfirst=True)
                                              ).dt.days
medication['end'] = (pd.to_datetime(medication['p_end'], dayfirst=True)-
                                           pd.to_datetime(medication['transplant_date'], dayfirst=True)
                                            ).dt.days
medication = medication.drop(columns=['p_start', 'p_end', 'transplant_date'])
print(f"Unique patients in clinical assessments: {medication['patient_id'].nunique()}")
print(f"Average entries per patient {len(medication) / medication['patient_id'].nunique():.1f}")

Unique patients in medication: 3335
Removing patients that are not in static_df
Unique patients in clinical assessments: 3296
Average entries per patient 78.9


In [6]:
medication

,patient_id,transplant_id,description,ddd,unit,atc,start,end
0,8905,9891,AEB071,200,mg,XXXXXXX,1,4.0
1,8905,9891,AEB071,200,mg,XXXXXXX,155,155.0
2,8905,9891,AEB071,400,mg,XXXXXXX,6,154.0
3,8905,9891,AEB071,300,mg,XXXXXXX,5,5.0
4,9086,10085,AEB071,400,mg,XXXXXXX,5,361.0
...,...,...,...,...,...,...,...,...
260213,14569,13283,Thymoglobuline,75,mg,L04AA04,0,1.0
260214,14656,13133,Thymoglobuline,150,mg,L04AA04,0,1.0
260215,12673,11660,Thymoglobuline,100,mg,L04AA04,0,1.0
260216,13684,12383,Thymoglobuline,125,mg,L04AA04,0,1.0
